In [1]:
import math
import urllib.parse
import urllib.request
import json
import webbrowser
import os

In [2]:
# 1. Chuyển địa chỉ sang tọa độ
def dia_chi_sang_toa_do(dia_chi):
    print(f" Đang tìm tọa độ trên bản đồ cho: {dia_chi}...")
    # Thêm "Ho Chi Minh, Vietnam" để giới hạn tìm kiếm chuẩn xác hơn trong khu vực
    query = f"{dia_chi}, Ho Chi Minh, Vietnam"
    url = f"https://nominatim.openstreetmap.org/search?q={urllib.parse.quote(query)}&format=json&limit=1"
    
    req = urllib.request.Request(url, headers={'User-Agent': 'FastFood_App/1.0'})
    try:
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())
            if data:
                return (float(data[0]['lat']), float(data[0]['lon']))
    except Exception as e:
        print("Lỗi kết nối bản đồ:", e)
    return None

In [3]:
# 2. TÌM ĐƯỜNG ĐI NGẮN NHẤT THỰC TẾ
def tim_duong_ngan_nhat_osrm(toa_do_quan, toa_do_khach):
    print(" Đang tính toán lộ trình đường đi ngắn nhất...")
    url = f"http://router.project-osrm.org/route/v1/driving/{toa_do_quan[1]},{toa_do_quan[0]};{toa_do_khach[1]},{toa_do_khach[0]}?overview=full&geometries=geojson"
    
    req = urllib.request.Request(url, headers={'User-Agent': 'FastFood_App/1.0'})
    try:
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())
            if data['code'] == 'Ok':
                km_thuc_te = round(data['routes'][0]['distance'] / 1000, 2)
                phut_thuc_te = round(data['routes'][0]['duration'] / 60, 1)
                
                route_coords = data['routes'][0]['geometry']['coordinates']
                toa_do_ve_duong = [[lat, lon] for lon, lat in route_coords]
                
                return toa_do_ve_duong, km_thuc_te, phut_thuc_te
    except Exception as e:
        print("Lỗi tìm đường:", e)
    return None, None, None

def trang_thai_giao_hang(km):
    return "Đang giao" if km <= 10 else "Đã hủy"

In [4]:
def tao_va_mo_ban_do(toa_do_quan, toa_do_khach, dia_chi_khach, km, phut, mang_toa_do_duong_di):
    duong_di_json = json.dumps(mang_toa_do_duong_di)
    
    # Cấu hình thời gian
    thoi_gian_lam_mon_giay = 5 * 60  # 5 phút cố định
    thoi_gian_giao_hang_giay = int(phut * 60)
    tong_giay_cho = thoi_gian_lam_mon_giay + thoi_gian_giao_hang_giay

    html_content = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>FastFood Universe - Tracking</title>
        <meta charset="utf-8" />
        <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
        <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
        <style>
            body {{ font-family: 'Segoe UI', Arial; background: #f8f9fa; margin: 0; padding: 20px; }}
            .container {{ max-width: 1000px; margin: 0 auto; }}
            
            .header-card {{ 
                background: white; padding: 20px; border-radius: 15px; 
                box-shadow: 0 4px 12px rgba(0,0,0,0.1); margin-bottom: 20px;
                display: flex; justify-content: space-between; align-items: center;
            }}

            .status-badge {{
                background: #fff4e6; color: #fd7e14; padding: 8px 15px;
                border-radius: 20px; font-weight: bold; font-size: 14px; margin-bottom: 10px;
                display: inline-block; border: 1px solid #ffe8cc;
            }}

            .timer-value {{ font-size: 32px; font-weight: 800; color: #e74c3c; }}
            
            #map {{ height: 500px; width: 100%; border-radius: 15px; box-shadow: 0 4px 12px rgba(0,0,0,0.1); }}
            
            .info-label {{ font-size: 12px; color: #868e96; text-transform: uppercase; }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="header-card">
                <div>
                    <div id="status-text" class="status-badge">👨‍🍳 Đang chế biến món ăn...</div>
                    <div class="info-label">Địa chỉ nhận hàng</div>
                    <div style="font-weight: bold; font-size: 16px;">{dia_chi_khach}</div>
                </div>
                <div style="text-align: right;">
                    <div class="info-label">Dự kiến nhận hàng sau</div>
                    <div id="countdown" class="timer-value">00:00</div>
                    <div class="info-label">Quãng đường: {km} km</div>
                </div>
            </div>
            <div id="map"></div>
        </div>

        <script>
            // Cấu hình bản đồ
            var map = L.map('map').setView([{toa_do_quan[0]}, {toa_do_quan[1]}], 15);
            L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png').addTo(map);

            var shopIcon = L.icon({{ iconUrl: 'https://cdn-icons-png.flaticon.com/512/3448/3448650.png', iconSize: [45, 45] }});
            var homeIcon = L.icon({{ iconUrl: 'https://cdn-icons-png.flaticon.com/512/1177/1177577.png', iconSize: [45, 45] }});

            L.marker([{toa_do_quan[0]}, {toa_do_quan[1]}], {{icon: shopIcon}}).addTo(map).bindPopup("Cửa hàng");
            L.marker([{toa_do_khach[0]}, {toa_do_khach[1]}], {{icon: homeIcon}}).addTo(map).bindPopup("Bạn ở đây");

            var routeCoords = {duong_di_json};
            L.polyline(routeCoords, {{ color: '#3498db', weight: 6, opacity: 0.6 }}).addTo(map);
            map.fitBounds(L.polyline(routeCoords).getBounds(), {{padding: [50, 50]}});

            // Logic đếm ngược đa giai đoạn
            let timeLeft = {tong_giay_cho};
            const timeCook = {thoi_gian_lam_mon_giay};
            
            function updateDisplay() {{
                const mins = Math.floor(timeLeft / 60);
                const secs = timeLeft % 60;
                document.getElementById('countdown').innerHTML = `${{mins}}p ${{secs < 10 ? '0' : ''}}${{secs}}s`;

                const statusBadge = document.getElementById('status-text');
                
                // Kiểm tra giai đoạn
                if (timeLeft > (timeLeft_initial - timeCook)) {{
                    statusBadge.innerHTML = "👨‍🍳 Đang chế biến món ăn...";
                    statusBadge.style.color = "#fd7e14";
                    statusBadge.style.background = "#fff4e6";
                }} else if (timeLeft > 0) {{
                    statusBadge.innerHTML = "🛵 Đang trên đường giao hàng...";
                    statusBadge.style.color = "#228be6";
                    statusBadge.style.background = "#e7f5ff";
                }} else {{
                    statusBadge.innerHTML = "✅ Món ăn đã sẵn sàng!";
                    statusBadge.style.color = "#40c057";
                    statusBadge.style.background = "#ebfbee";
                    clearInterval(timerId);
                }}
                timeLeft--;
            }}

            const timeLeft_initial = timeLeft;
            const timerId = setInterval(updateDisplay, 1000);
            updateDisplay();
        </script>
    </body>
    </html>
    """
    # Ghi file và mở trình duyệt (giống như cũ)
    file_name = "fastfood_tracking.html"
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(html_content)
    webbrowser.open('file://' + os.path.realpath(file_name))

In [5]:
# CHƯƠNG TRÌNH CHÍNH
toa_do_quan = (10.7634, 106.6821)  # FastFood Universe (227 Nguyễn Văn Cừ, Quận 5)
print(" CHÀO MỪNG ĐẾN VỚI FASTFOOD UNIVERSE")
print(" Lưu ý: Cửa hàng chỉ giao trong phạm vi tối đa 10km.")
print("-" * 50)

# 1. NHẬP ĐỊA CHỈ TỰ DO
dia_chi_khach = input(" Nhập địa chỉ nhận hàng của bạn (VD: Chợ Bến Thành, Landmark 81, 123 Lê Lợi...): ").strip()
toa_do_khach = dia_chi_sang_toa_do(dia_chi_khach)

if not toa_do_khach:
    print(" Lỗi: Không tìm được địa chỉ trên bản đồ. Vui lòng thử nhập chi tiết hơn (gồm số nhà, tên đường, phường, quận).")
else:
    duong_di, km_thuc, phut_thuc = tim_duong_ngan_nhat_osrm(toa_do_quan, toa_do_khach)

    if not duong_di:
        print(" Lỗi: Không tìm được lộ trình giao thông đến địa chỉ này.")
    else:
        # 2. KIỂM TRA ĐIỀU KIỆN 10KM (Sử dụng khoảng cách thực tế)
        if km_thuc > 10:
            print(f"\n TỪ CHỐI ĐƠN HÀNG:")
            print(f"Khoảng cách đến chỗ bạn là {km_thuc} km. Rất tiếc, cửa hàng chỉ giao trong phạm vi 10 km đổ lại để đảm bảo chất lượng món ăn!")
        else:
            # 3. NẾU <= 10KM THÌ TIẾN HÀNH ĐẶT ĐƠN VÀ VẼ BẢN ĐỒ
            ket_qua = {
                "ten_quan": "FastFood Universe",
                "dia_chi_khach": dia_chi_khach,
                "khoang_cach_thuc_te_km": km_thuc,
                "thoi_gian_lai_xe_phut": phut_thuc,
                "trang_thai": trang_thai_giao_hang(km_thuc)
            }
            
            print("\n THÔNG TIN LỘ TRÌNH (ĐỦ ĐIỀU KIỆN GIAO):")
            print(json.dumps(ket_qua, ensure_ascii=False, indent=4))
            tao_va_mo_ban_do(toa_do_quan, toa_do_khach, dia_chi_khach, km_thuc, phut_thuc, duong_di)

 CHÀO MỪNG ĐẾN VỚI FASTFOOD UNIVERSE
 Lưu ý: Cửa hàng chỉ giao trong phạm vi tối đa 10km.
--------------------------------------------------
 Đang tìm tọa độ trên bản đồ cho: 236 le van sy...
 Đang tính toán lộ trình đường đi ngắn nhất...

 THÔNG TIN LỘ TRÌNH (ĐỦ ĐIỀU KIỆN GIAO):
{
    "ten_quan": "FastFood Universe",
    "dia_chi_khach": "236 le van sy",
    "khoang_cach_thuc_te_km": 5.66,
    "thoi_gian_lai_xe_phut": 7.6,
    "trang_thai": "Đang giao"
}
